<a href="https://colab.research.google.com/github/siva2513-ship-it/paddy_disease_detection/blob/main/colab_notebooks/Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastai

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from fastai.vision.all import *
from pathlib import Path

model_path = Path(
    '/content/drive/MyDrive/Paddy_Disease_Project/resnet34_paddy_baseline.pkl'
)

learn = load_learner(model_path)

print("Model loaded successfully!")
print("Classes:", learn.dls.vocab)
print("Number of classes:", len(learn.dls.vocab))

Model loaded successfully!
Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']
Number of classes: 10


In [6]:
print(learn.model)

Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [7]:
from pathlib import Path

train_path = Path('/content/paddy-doctor/paddy-disease-classification/train_images')

print("Dataset exists:", train_path.exists())

Dataset exists: False


In [8]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [9]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [10]:
!kaggle datasets download -d imbikramsaha/paddy-doctor

Dataset URL: https://www.kaggle.com/datasets/imbikramsaha/paddy-doctor
License(s): CC0-1.0
100% 1.02G/1.02G [00:14<00:00, 75.0MB/s]



In [11]:
!unzip -q paddy-doctor.zip -d /content/paddy-doctor

In [12]:
from pathlib import Path

train_path = Path('/content/paddy-doctor/paddy-disease-classification/train_images')

print("Dataset exists:", train_path.exists())
print("Classes:", sorted([p.name for p in train_path.iterdir() if p.is_dir()]))

Dataset exists: True
Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']


In [13]:
from fastai.vision.all import *

dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(0.2, seed=42),
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

dls = dblock.dataloaders(train_path)

print("Classes:", dls.vocab)
print("Number of classes:", len(dls.vocab))

Classes: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']
Number of classes: 10


In [14]:
print("Training images:", len(dls.train_ds))
print("Validation images:", len(dls.valid_ds))

Training images: 8326
Validation images: 2081


In [15]:
dls.vocab

['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']

In [16]:
learn.dls.vocab

['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']

In [17]:
learn.dls = dls

In [19]:
print(learn.dls.train)
print(learn.dls.valid)

In [20]:
import torch
import numpy as np
from tqdm.auto import tqdm

# Put model in evaluation mode
learn.model.eval()

# We want everything BEFORE the final 10-class Linear layer
feature_extractor = torch.nn.Sequential(
    learn.model[0],
    *list(learn.model[1])[:-1]
)

feature_extractor = feature_extractor.to(learn.dls.device)
feature_extractor.eval()

def extract_features(dl):
    features = []
    labels = []

    with torch.no_grad():
        for xb, yb in tqdm(dl):
            xb = xb.to(learn.dls.device)

            # Forward through ResNet34 + 512-D representation
            feats = feature_extractor(xb)

            features.append(feats.cpu().numpy())
            labels.append(yb.cpu().numpy())

    return np.concatenate(features), np.concatenate(labels)

In [23]:
train_dl_full = learn.dls.train.new(
    shuffle=False,
    drop_last=False
)

print("Expected training images:", len(learn.dls.train_ds))
print("Feature extraction images:", len(train_dl_full.dataset))

Expected training images: 8326
Feature extraction images: 8326


In [24]:
X_train, y_train = extract_features(train_dl_full)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

  0%|          | 0/131 [00:00<?, ?it/s]

X_train shape: (8326, 512)
y_train shape: (8326,)


In [25]:
valid_dl_full = learn.dls.valid.new(
    shuffle=False,
    drop_last=False
)

In [27]:
X_valid, y_valid = extract_features(valid_dl_full)

print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

  0%|          | 0/33 [00:00<?, ?it/s]

X_valid shape: (2081, 512)
y_valid shape: (2081,)


In [28]:
import numpy as np
from pathlib import Path

features_path = Path(
    '/content/drive/MyDrive/Paddy_Disease_Project/features'
)
features_path.mkdir(parents=True, exist_ok=True)

# Save features and labels
np.save(features_path / 'X_train.npy', X_train)
np.save(features_path / 'y_train.npy', y_train)
np.save(features_path / 'X_valid.npy', X_valid)
np.save(features_path / 'y_valid.npy', y_valid)

print("✅ Features saved to Google Drive!")

✅ Features saved to Google Drive!
